# Enformer Regulatory Activity Prediction

![Enformer Regulatory Activity Prediction](https://proto-bio.github.io/proto-assets/images/tool/enformer/hero.png)

Enformer is a deep learning model for predicting gene regulatory activity directly from DNA sequence, published by Avsec et al. (2021). It processes 196,608 bp input windows and outputs binned predictions across 896 spatial bins for thousands of regulatory tracks covering chromatin accessibility, histone modifications, and gene expression. The model supports both human and mouse output heads, and track selection lets you extract only the signals relevant to your analysis.

This notebook demonstrates how to predict regulatory activity from a DNA sequence and how to perform a variant-effect comparison by running reference and alternate allele windows through the model and computing log2 fold-change.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("enformer")
display_overview("enformer")
display_docs_section("enformer", "Background")

# Enformer

[Enformer](https://github.com/google-deepmind/deepmind-research/tree/master/enformer) is a deep learning model that predicts [gene expression](https://en.wikipedia.org/wiki/Gene_expression) and chromatin activity directly from DNA sequence, developed by Google DeepMind and Calico Life Sciences. It reads a fixed 196,608 base-pair input window and predicts activity across thousands of genomic measurement tracks, summarized over 896 output bins of 128 base pairs each. This toolkit exposes Enformer through a single registered tool that returns the predicted track activity for one or more sequences.

Enformer ([Avsec et al., 2021](https://doi.org/10.1038/s41592-021-01252-x)) is a neural network that predicts [gene expression](https://en.wikipedia.org/wiki/Gene_expression) and chromatin state from genomic DNA sequence. Its architecture combines convolutional layers with [transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) self-attention, which allows the model to integrate the influence of distal [cis-regulatory elements](https://en.wikipedia.org/wiki/Cis-regulatory_element) such as [enhancers](https://en.wikipedia.org/wiki/Enhancer_(genetics)) located up to 100 kilobases away from a [promoter](https://en.wikipedia.org/wiki/Promoter_(genetics)). The published work reports that this long-range modeling substantially improves gene expression prediction accuracy relative to earlier convolutional models, and that it yields more accurate predictions of the effect of genetic variants on expression for both natural variants and saturation mutagenesis measured by reporter assays.

Enformer predicts activity across 896 output bins, each summarizing 128 base pairs, for a large panel of functional genomics assays. The human output head covers 5,313 tracks spanning [chromatin accessibility](https://en.wikipedia.org/wiki/DNase_I_hypersensitive_site), [transcription factor](https://en.wikipedia.org/wiki/Transcription_factor) binding, [histone modifications](https://en.wikipedia.org/wiki/Histone), and [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_gene_expression) expression measurements, and the mouse output head covers 1,643 tracks. Because the model maps sequence directly to these signals, it can be used to compare the predicted regulatory activity of alternative alleles at the same locus, which is the basis for its use in interpreting noncoding genetic variation. The published analysis additionally shows that Enformer learns enhancer-promoter relationships directly from the input sequence.

## Available tools

In [2]:
display_available_tools("enformer")

- **`run_enformer_gradient()`** — Compute Enformer track-and-bin activity gradients for relaxed DNA positions
- **`run_enformer()`** — Gene expression and regulatory activity prediction using Enformer

### `run_enformer`

Enformer predicts regulatory activity from a 196,608 bp DNA input window, returning a signal matrix of shape [896, num_tracks] where each of the 896 spatial bins covers a 128 bp window. The `output_tracks` parameter selects which of the thousands of regulatory tracks to extract, and the `species` parameter switches between human and mouse output heads. A common use is variant-effect analysis: running the same input window with reference and alternate alleles and comparing the resulting prediction matrices to identify regulatory impact.

In [3]:
import numpy as np
from proto_tools import ENFORMER_CONTEXT, EnformerConfig, EnformerInput, run_enformer

In [4]:
# Display input docs
display_api_reference("enformer", "input", "run_enformer")

from proto_tools import NCBIEfetchConfig, NCBIEfetchInput, run_ncbi_efetch

# Fetch a real human genomic window on chr11 centred on the HBB locus, exactly the
# length Enformer requires, instead of a synthetic repeat.
_center = 5227430  # HBB midpoint, GRCh38 chr11 (NC_000011.10)
_fetch = run_ncbi_efetch(
    NCBIEfetchInput(
        db="nuccore", identifier="NC_000011.10", return_format="fasta",
        seq_start=_center - ENFORMER_CONTEXT // 2, seq_stop=_center + ENFORMER_CONTEXT // 2,
    ),
    NCBIEfetchConfig(),
)
sequence = _fetch.fasta_records[0].sequence.upper()[:ENFORMER_CONTEXT]
inputs = EnformerInput(sequences=[sequence])

**Input** — `EnformerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[SequenceWindow]</code> | required | Sequence(s) to score; each a DNA window with optional target_range (a bare string is accepted) |

In [5]:
# Display config docs
display_api_reference("enformer", "config", "run_enformer")

config = EnformerConfig(
    output_tracks=[0, 1, 2],
    species="human",
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `EnformerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>output_tracks</code> | <code>list[int]</code> | <code>[0]</code> | Indices of Enformer output tracks to extract (5313 human / 1643 mouse) |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species track head to use |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |

In [6]:
# Run the prediction function
pred_result = run_enformer(inputs, config)

Running run_enformer [00:00]

In [7]:
display_api_reference("enformer", "output", "run_enformer")

print(f"Output bins: {len(pred_result.results[0].prediction)}")
print(f"Tracks per bin: {len(pred_result.results[0].prediction[0])}")

**Output** — `EnformerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[EnformerPredictionResult]</code> | required | Per-sequence Enformer prediction results |
| <code>output_tracks</code> | <code>list[int]</code> | required | Track indices extracted from Enformer |
| <code>species</code> | <code>str</code> | required | Species used for prediction ('human' or 'mouse') |

**`EnformerPredictionResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequence</code> | <code>str</code> | required | DNA sequence originally provided to the tool |
| <code>sequence_length</code> | <code>int</code> | required | Length of the provided DNA sequence |
| <code>prediction</code> | <code>list[list[float]]</code> | required | Predicted track activity per 128 bp bin (shape [896, num_tracks]) |
| <code>context_start</code> | <code>int</code> | required | 0-based start of the Enformer input window in sequence |
| <code>context_end</code> | <code>int</code> | required | 0-based exclusive end of the Enformer input window |
| <code>output_start</code> | <code>int</code> | required | 0-based start of the span covered by Enformer output bins |
| <code>output_end</code> | <code>int</code> | required | 0-based exclusive end of the Enformer output-bin span |
| <code>output_resolution</code> | <code>int</code> | <code>128</code> | Base pairs represented by each output bin |
| <code>target_start</code> | <code>int &#124; None</code> | <code>None</code> | Requested target start, if target_ranges was provided |
| <code>target_end</code> | <code>int &#124; None</code> | <code>None</code> | Requested target end, if target_ranges was provided |

Output bins: 896
Tracks per bin: 3


#### Variant-Effect Comparison

A common use of Enformer is comparing predictions between a reference and an alternate allele to identify regulatory variants. We introduce a single-nucleotide change at the center of the input window, run both sequences through the model, and compute log2 fold-change across all bins and tracks. Large absolute fold-change values indicate positions and tracks where the variant has a meaningful effect on predicted regulatory activity.

In [8]:
# Introduce a single-nucleotide substitution at the center of the input window
center = ENFORMER_CONTEXT // 2
ref_sequence = sequence
alt_base = "A" if ref_sequence[center] != "A" else "C"
alt_sequence = ref_sequence[:center] + alt_base + ref_sequence[center + 1:]

ref_result = run_enformer(EnformerInput(sequences=[ref_sequence]), config)
alt_result = run_enformer(EnformerInput(sequences=[alt_sequence]), config)

ref_pred = np.array(ref_result.results[0].prediction)
alt_pred = np.array(alt_result.results[0].prediction)
log2_fc = np.log2((alt_pred + 1e-6) / (ref_pred + 1e-6))
print(f"Max |log2 FC|: {np.abs(log2_fc).max():.4f}")

Running run_enformer [00:00]

Running run_enformer [00:00]

Max |log2 FC|: 0.0606


### `run_enformer_gradient`

The gradient tool turns Enformer's track predictions into a differentiable objective. Each term names an output track, a span of 128 bp output bins, a direction, and a weight; the terms are signed so the summed objective is always minimized, and the gradient comes back with respect to a relaxed distribution over the bases being designed.

Enformer reads a fixed 196,608 bp context, but a design problem usually varies only a small part of it. A candidate therefore bundles the **whole** context sequence with the 1-indexed `positions` whose bases are relaxed and a `(len(positions), 4)` logits matrix aligned to them — so the gradient covers only the designed positions while the surrounding context still reaches the model.

Where the variant-effect comparison above asks *what does this substitution do*, the gradient answers *which substitution would help* — it is the same model read as a design objective rather than a predictor.

In [9]:
from proto_tools import (
    ENFORMER_CONTEXT, EnformerGradientCandidate, EnformerGradientConfig,
    EnformerGradientInput, EnformerGradientLossTerm, run_enformer_gradient,
)

display_api_reference("enformer", "input", "run_enformer_gradient")

# Design a 24 bp stretch at the centre of the same window used above, where
# Enformer's output bins are densest. Positions are 1-indexed and inclusive.
grad_positions = [ENFORMER_CONTEXT // 2 + offset for offset in range(24)]
grad_candidate = EnformerGradientCandidate(
    sequence=sequence,
    positions=grad_positions,
    # Start from a flat distribution over A, C, G, T at each designed position.
    logits=[[0.25, 0.25, 0.25, 0.25]] * len(grad_positions),
)


**Input** — `EnformerGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[EnformerGradientCandidate]</code> | required | One context window with relaxed positions per candidate; a single bundle is accepted alone |

In [10]:
display_api_reference("enformer", "config", "run_enformer_gradient")

# Push track 0 up over the output bins covering the designed window.
grad_config = EnformerGradientConfig(
    loss_terms=[EnformerGradientLossTerm(track=0, bin_start=440, bin_end=456, direction="max")],
)

grad_output = run_enformer_gradient(EnformerGradientInput(inputs=grad_candidate), grad_config)


**Config** — `EnformerGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend the raw input matrix (0) toward softmax probabilities (1). |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>loss_terms</code> | <code>list[EnformerGradientLossTerm]</code> | <code>[EnformerGradientLossTerm(track=0, bin_start=1, bin_end=896, direction='max', weight=1.0)]</code> | Track-and-bin objective terms summed into the scalar loss |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

Running run_enformer_gradient [00:00]

In [11]:
display_api_reference("enformer", "output", "run_enformer_gradient")

# One result per candidate; this call passed one.
grad_result = grad_output.results[0]
print(f"Weighted objective: {grad_result.loss:.4f}")
print(f"Designed positions: {grad_result.metrics['designed_positions']}")
print(f"Gradient shape:     ({len(grad_result.gradient)}, {len(grad_result.gradient[0])})")
for grad_term in grad_result.metrics["loss_terms"]:
    print(f"  track {grad_term['track']} bins {grad_term['bin_start']}-{grad_term['bin_end']}: "
          f"mean activity {grad_term['mean_activity']:.4f} ({grad_term['direction']})")
display_api_reference("enformer", "input", "run_enformer_gradient")
display_api_reference("enformer", "config", "run_enformer_gradient")
display_api_reference("enformer", "output", "run_enformer_gradient")


**Output** — `EnformerGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GradientResult]</code> | required | One result per input item, in input order, each carrying its own gradient, loss and metrics |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`GradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Gradient of this item's loss w.r.t. its input logits; matches its shape and column order |
| <code>loss</code> | <code>float</code> | required | This item's objective; lower is better (typically a positive negative log-likelihood) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**Metrics** — `EnformerGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>loss</code> **(primary)** | <code>float</code> |  |  | Weighted scalar Enformer objective value for one candidate. |
| <code>designed_positions</code> | <code>int</code> | <code>&gt;= 1</code> |  | Number of relaxed context positions this candidate carries. |

Weighted objective: -0.0650
Designed positions: 24
Gradient shape:     (24, 4)
  track 0 bins 440-456: mean activity 0.0650 (max)


**Input** — `EnformerGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[EnformerGradientCandidate]</code> | required | One context window with relaxed positions per candidate; a single bundle is accepted alone |

**Config** — `EnformerGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend the raw input matrix (0) toward softmax probabilities (1). |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>loss_terms</code> | <code>list[EnformerGradientLossTerm]</code> | <code>[EnformerGradientLossTerm(track=0, bin_start=1, bin_end=896, direction='max', weight=1.0)]</code> | Track-and-bin objective terms summed into the scalar loss |
| <code>species</code> | <code>Literal['human', 'mouse']</code> | <code>'human'</code> | Species model to use |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring |

**Output** — `EnformerGradientOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[GradientResult]</code> | required | One result per input item, in input order, each carrying its own gradient, loss and metrics |
| <code>vocab</code> | <code>list[str]</code> | required | Symbols defining the column ordering for both the input logits and the returned gradient. |

**`GradientResult`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>gradient</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Gradient of this item's loss w.r.t. its input logits; matches its shape and column order |
| <code>loss</code> | <code>float</code> | required | This item's objective; lower is better (typically a positive negative log-likelihood) |
| <code>metrics</code> | <code>dict[str, Any]</code> | <code>{}</code> | Metrics reported for this item alongside its objective value. |

**Metrics** — `EnformerGradientSampleMetrics` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>loss</code> **(primary)** | <code>float</code> |  |  | Weighted scalar Enformer objective value for one candidate. |
| <code>designed_positions</code> | <code>int</code> | <code>&gt;= 1</code> |  | Number of relaxed context positions this candidate carries. |

## Export Results

Prediction outputs can be saved to standard file formats for downstream analysis or integration with other computational pipelines. Supported formats include JSON and CSV.

In [12]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

pred_result.export(name="enformer_prediction", export_path=output_dir, file_format="json")
print(f"Prediction exported to {output_dir}/enformer_prediction.json")

ref_result.export(name="enformer_ref", export_path=output_dir, file_format="csv")
print(f"Reference summary exported to {output_dir}/enformer_ref.csv")

Prediction exported to example_output/enformer_prediction.json
Reference summary exported to example_output/enformer_ref.csv
